# ANOMALY DETECTION, THEORY
Anomaly Detection (AD) is the process of identifying data points, events, or patterns that deviate significantly from expected behavior.

In **unsupervised AD**, the goal is to detect anomalies without labeled data – the model learns the normal patterns and flags deviations (==> We will mainly focus on this!)

In **supervised AD**, instances of anomalies are available, models learn to separate normal patterns from anomalies. 

AD algorithms typically produce, for each sample, an **anomaly score** indicating how likely it is to be an anomaly. A threshold is then applied to classify samples as normal or anomalous.

---
---

## Types of Anomalies

### Point Anomalies
Individual data points that significantly differ from the rest of the population. 

Example: a person withdrawing $10,000 in cash, at 3 AM from an ATM. 



In [1]:
import numpy as np
from sklearn.datasets import make_blobs
import matplotlib.pyplot as plt


In [2]:
# add arrow to an anomalous point
def show_anomaly(ax, point):
    # white background, black border for the text box
    ax.annotate('Anomaly', xy=point, xytext=point - (0.5, 0.5), arrowprops=dict(facecolor='red', shrink=0.05), fontsize=12, color='red', bbox=dict(boxstyle="round,pad=0.3", edgecolor='black', facecolor='white'))

In [ ]:
# generate dataset (sampled from normal distribution, with 1 anomaly)
X_normal = np.random.normal(loc=0.0, scale=0.25, size=(300, 2))
X_anomaly = np.array([[2,2]])
X = np.vstack([X_normal, X_anomaly])

# plot dataset
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(X[:, 0], X[:, 1], c='c', edgecolors='k')
show_anomaly(ax, X_anomaly[0])

ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.grid()
ax.set_axisbelow(True)

---

### Contextual Anomalies

For contextual anomalies, a data point that is anomalous only within a certain context i.e., it would be "normal" in other circumstances. 

Example: a person doing a wire transfer for $500,000. If it was a company, the operation would be less anomalous!

In [ ]:
X, y = make_blobs(n_samples=100, centers=3, cluster_std=0.5, random_state=1)
X *= .1
anomaly_id = (y == 0).nonzero()[0][-1]
y[anomaly_id] = 1

markers = ["P", "o", "^"]
group_labels = ["Cats", "Dogs", "Parrots"]
fig, ax = plt.subplots(figsize=(5, 4))
for c_id in np.unique(y):
    ax.scatter(X[y==c_id, 0], X[y==c_id, 1], label=group_labels[c_id], edgecolors='k', marker=markers[c_id], s=75)

show_anomaly(ax, X[anomaly_id])

ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.legend()
ax.grid()
ax.set_axisbelow(True)

---
### Collective Anomalies

A group of otherwise normal points that collectively form an anomaly.

Example: multiple people depositing cash for $9,000 and then wire-transferring the same amount to a foreign account. A single deposit of $9,000 is not anomalous, but the collective behavior is suspicious.

In [ ]:
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=900, noise=0.1, random_state=1)
X_noise = np.random.normal(loc=(1.5,0.75), scale=0.075, size=(20, 2))

X = np.vstack([X, X_noise])

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(X[:, 0], X[:, 1], c='c', edgecolors='k')

show_anomaly(ax, X_noise[0])

ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.grid()
ax.set_axisbelow(True)

--- 
---

## Anomaly Detection techniques

* **Statistical methods**:
  * Based on statistical models to identify data points that deviate significantly from the expected distribution
  * Example: Z-score method
* **Clustering-based methods**:
  * Use clustering algorithms to group similar data points together and identify those that do not belong to any cluster as anomalies
  * Example: DBSCAN, K-means
* **Proximity-based methods**:
  * Use distance or density measures to identify anomalies based on their proximity to other data points
  * Example: k-NN, Local Outlier Factor (LOF)
* **Tree-based methods**:
  * Use properties of trees to isolate anomalies in the data
  * Example: Isolation Forest
* **Boundary-based methods**:
  * Use decision boundaries to separate normal data points from anomalies
  * Example: One-Class SVM
* **Neural Network-based methods**:
  * Use neural networks to learn data distributions and identify anomalies (e.g., based on reconstruction errors)
  * Example: Autoencoders

---
> ### Statistical methods

Quantify the deviation of a data point from the expected distribution using statistical measures. For instance, the Z-score method calculates how many standard deviations a data point is from the mean. Points with high absolute Z-scores are considered anomalies.


Multivariate generalizations can also be used (e.g., Mahalanobis distance).

In [6]:
heights = np.random.normal(loc=170, scale=10, size=1000)
heights_anomalies = np.array([250, 260, 270])
heights = np.hstack([heights, heights_anomalies])

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.hist(heights, bins=30, color='c', edgecolor='k')
ax.set_xlabel('Height (cm)')
ax.set_ylabel('Frequency')
ax.grid()
ax.set_axisbelow(True)

In [ ]:
z_normalized = (heights - np.mean(heights)) / np.std(heights)

anomaly_threshold = 3

print(f"Anomalies (3σ): {heights[abs(z_normalized) > anomaly_threshold]}")

abs(z_normalized) is the distance from the mean measured in units of standard deviation.
Because you defined:

$z = \frac{x - \mu}{\sigma}$

- x − μ is the raw distance from the mean (in cm here).
- Dividing by σ rescales that distance by “how big a typical spread is”.

This number answers:  
> “how many standard deviations away is x?”  
> That’s why the threshold “3” means 3 standard deviations  

Equivalently:

$|z| > 3 \quad \Longleftrightarrow \quad \frac{|x-\mu|}{\sigma} > 3 \quad \Longleftrightarrow \quad |x-\mu| > 3\sigma$

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.hist(z_normalized, bins=30, color='c', edgecolor='k')
ax.set_xlabel('Normalized Height (Z-score)')
ax.set_ylabel('Frequency')
ax.grid()
ax.set_axisbelow(True)

ax.axvline(x=anomaly_threshold, color='r', linestyle='--', label=f'Anomaly Threshold (+{anomaly_threshold}σ)')

xlim = ax.get_xlim()
ax.axvspan(anomaly_threshold, xlim[1], color='red', alpha=0.3)
ax.legend()
ax.set_xlim(xlim)

---
> ### Clustering-based methods

Clustering algorithms group similar data points together. Data points that do not belong to any cluster or are in small clusters are considered anomalies.

> #### K-means
With K-means, for example, points that are far from their assigned cluster centroids can be flagged as anomalies.


In [ ]:
X_blob, y_blob = make_blobs(n_samples=1_000, centers=5, cluster_std=0.5, random_state=1)
X_anomalies = np.random.uniform(X_blob.min(), X_blob.max(), size=(50, 2))

X_blob = np.vstack([X_blob, X_anomalies])

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(X_blob[:, 0], X_blob[:, 1], c='c', edgecolors='k')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.grid()
ax.set_axisbelow(True)

In [11]:
from sklearn.cluster import KMeans

model = KMeans(n_clusters=5)
model.fit(X_blob)

# transform() computes the distance between each point and each cluster center
# (equivalent to euclidean_distances(X, model.cluster_centers_))
# Each point is assigned to the closest cluster. With min(axis=1) we can extract that distance
distances = model.transform(X_blob).min(axis=1)

fraction_of_anomalies = 0.05 # we expect ~ 5% of the data to be anomalies

anomalous_indices = distances.argsort()[-int(fraction_of_anomalies * len(X_blob)):]
y_anomaly = np.zeros(len(X_blob), dtype=int)
y_anomaly[anomalous_indices] = 1

After KMeans is fit, it has 5 cluster centers.
transform(X_blob) returns, for each point, its distance to each center.

So the output shape is:  
- n_points × n_clusters  
Here: 1050 × 5  

Row i looks like:  
[dist(point_i, center_0),  
 dist(point_i, center_1),  
 dist(point_i, center_2),  
 dist(point_i, center_3),  
 dist(point_i, center_4)]  

Then .min(axis=1) takes the smallest distance in each row → i.e. distance to the nearest cluster center. So:  
- For every point, compute how far it is from its closest cluster center.  
- Big distance = point doesn’t belong well to any cluster = suspicious.  


You decide: “I expect about 5% anomalies.”  
So you:  
1.	compute a “weirdness score” for every point = distance to nearest center  
2.	sort points by that score  
3.	mark the top 5% farthest points as anomalies  

> anomalous_indices = distances.argsort()[-int(fraction_of_anomalies * len(X_blob)):]  
- distances.argsort() gives indices that would sort distances from smallest to largest.
- early indices = points close to a center (normal)
- late indices = points far from all centers (weird)
- [-k:] takes the last k indices → the k most distant points, where k = 0.05 * N.

It's like doing:
> fraction_of_anomalies * len(X_blob)
- number of anomalies we're expecting K

> distances.argsort()
- sort the distances from biggest to smallest and pick the corresponding index

> distances.argsort()[-K:]
- pick the last K indexes which are the index of the K points most further away to their centroids

In plain English:  
> “Pick the indices of the 5% points that are farthest from their closest cluster.”

### This approach is so fucking dumb, other approach
1.	compute each point’s distance to its nearest centroid  
2.	sort those distances ascending  
3.	plot the sorted distances -->  --> on the left we'll have points that are very close to their centroids, on the right we'll have points further away from their centroids (anomalies)  
4.	pick a cutoff at the knee: “before knee = normal, after knee = anomalies” --> figure out the number of anomalies  


> Still an heuristic but at least you're not just randomly guessing the % of anomalies ...

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.hist(distances, bins=30, color='c', edgecolor='k')
ax.set_xlabel('Distance to Nearest Cluster Center')
ax.set_ylabel('Frequency')
ax.grid()
ax.set_axisbelow(True)

In [ ]:
# my plot ot avoid that assumption 5% bullshit

sorted_distances_to_closest_centroid = np.sort(distances)
plt.figure(figsize=(8,6))
plt.plot(sorted_distances_to_closest_centroid)
plt.axvline(x=950, c='red', alpha = 0.3)
plt.xlabel('sorted points based on distance')
plt.ylabel('sorted distances to closest centroids')
plt.show()


> #### DBSCAN
Some clustering algorithms (e.g., DBSCAN) can directly identify outliers as points that do not belong to any cluster.

In scikit-learn, these points are labeled with cluster -1.

In [ ]:
from sklearn.cluster import DBSCAN
import pandas as pd

df = pd.read_csv("chameleon.data", sep=" ", header=None, names=['x1', 'x2'])
X = df.values

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(X[:, 0], X[:, 1], c='c', edgecolors='k')

# FOR DBSCAN THIS IS SO SMART TO SPOT NOISE --> MASKING + SLICING

In [ ]:
model = DBSCAN(eps=15, min_samples=5)
y_pred = model.fit_predict(X)

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(X[y_pred>-1, 0], X[y_pred>-1, 1], c='c', edgecolors='k')
ax.scatter(X[y_pred==-1, 0], X[y_pred==-1, 1], c='r', marker='x')

ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.grid()
ax.set_axisbelow(True)

---

### Proximity-based methods

These methods rely on the distance or density of data points. Points that are far from others or in low-density regions are considered anomalies.

> #### Local Outlier Factor (LOF)

In [ ]:
from sklearn.neighbors import LocalOutlierFactor

lof = LocalOutlierFactor(n_neighbors=10, contamination=0.05) # contamination: expected fraction of outliers in the data (will be used to define the threshold)
y_anomaly = lof.fit_predict(X)

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(X[y_anomaly==1, 0], X[y_anomaly==1, 1], c='c', edgecolors='k')
ax.scatter(X[y_anomaly==-1, 0], X[y_anomaly==-1, 1], c='r', marker='x')


The reults are similar to those obtained with DBSCAN, since the "high density" regions all have approximately the same density. What happens if some region of space becomes less dense?

In [ ]:
# find the biggest cluster (according to DBSCAN), and downsample it (with [::3] we are selecting approx. 1/3 of the dataset)

model = DBSCAN(eps=15, min_samples=5)
y_pred = model.fit_predict(X)

clusters, frequencies = np.unique(y_pred, return_counts=True)

biggest_cluster = clusters[frequencies.argmax()]
X_new = np.vstack([X[y_pred != biggest_cluster], X[y_pred == biggest_cluster][::3]])

# now, let's apply DBSCAN once again and LOF to the new dataset (with one cluster downsampled) and let's compare the results.

fig, ax = plt.subplots(1, 4, figsize=(20, 4), sharey=True)

ax[0].scatter(X[:, 0], X[:, 1], c='c', edgecolors='k')
ax[0].set_xlabel('Feature 1')
ax[0].set_ylabel('Feature 2')
ax[0].grid()
ax[0].set_axisbelow(True)

ax[1].scatter(X_new[:, 0], X_new[:, 1], c='c', edgecolors='k')
ax[1].set_xlabel('Feature 1')
ax[1].set_ylabel('Feature 2')
ax[1].grid()
ax[1].set_axisbelow(True)

# DBSCAN
model = DBSCAN(eps=15, min_samples=5)
y_pred = model.fit_predict(X_new)

ax[2].scatter(X_new[y_pred>-1, 0], X_new[y_pred>-1, 1], c='c', edgecolors='k')
ax[2].scatter(X_new[y_pred==-1, 0], X_new[y_pred==-1, 1], c='r', marker='x')
ax[2].set_xlabel('Feature 1')
ax[2].set_ylabel('Feature 2')
ax[2].grid()
ax[2].set_axisbelow(True)

# LOF
lof = LocalOutlierFactor(n_neighbors=10, contamination=0.05) # contamination: expected fraction of outliers in the data (will be used to define the threshold)
y_anomaly = lof.fit_predict(X_new)


ax[3].scatter(X_new[y_anomaly==1, 0], X_new[y_anomaly==1, 1], c='c', edgecolors='k')
ax[3].scatter(X_new[y_anomaly==-1, 0], X_new[y_anomaly==-1, 1], c='r', marker='x')
ax[3].set_xlabel('Feature 1')
ax[3].set_ylabel('Feature 2')
ax[3].grid()
ax[3].set_axisbelow(True)


ax[1].set_title('Original dataset')
ax[1].set_title('Dataset with downsampled cluster')
ax[2].set_title('DBSCAN Anomalies')
ax[3].set_title('LOF Anomalies')


---

> ### Tree-based methods

Isolation Forests are ensemble methods that isolate anomalies by randomly partitioning the data using random decision trees. Anomalies are easier to isolate and thus have shorter average path lengths in the trees.

> Isolation Forest is basically “find weird points by seeing how quickly you can separate them from everyone else using random cuts”.

### Why anomalies get isolated fast?
- Normal points live in dense regions with many neighbors around them. Random cuts usually keep them grouped with others for a while → you need many splits to isolate one specific normal point.
- Anomalies are “lonely” (far from others or in sparse regions). A few random cuts are enough to separate them from the pack → they get isolated in fewer splits.

In [ ]:
from sklearn.ensemble import IsolationForest


fig, ax = plt.subplots(2, 2, figsize=(10, 8))

isoforest = IsolationForest(n_estimators=100, contamination=0.05, random_state=1)
y_anomaly = isoforest.fit_predict(X_blob)

ax[0, 0].scatter(X_blob[y_anomaly==1, 0], X_blob[y_anomaly==1, 1], c='c', edgecolors='k')
ax[0, 0].scatter(X_blob[y_anomaly==-1, 0], X_blob[y_anomaly==-1, 1], c='r', marker='x')


# Heatmap of anomaly scores
ax[1,0].scatter(X_blob[:, 0], X_blob[:, 1], c=isoforest.score_samples(X_blob))

XX, YY = np.meshgrid(np.linspace(X_blob[:,0].min(), X_blob[:,0].max(), 100),
                     np.linspace(X_blob[:,1].min(), X_blob[:,1].max(), 100))
Z = isoforest.score_samples(np.c_[XX.ravel(), YY.ravel()]).reshape(XX.shape)
Z = (Z - Z.min()) / (Z.max() - Z.min())
cm = ax[1,0].contourf(XX, YY, Z, levels=50, cmap='RdBu')


isoforest = IsolationForest(n_estimators=100, contamination=0.05, random_state=1)
y_anomaly = isoforest.fit_predict(X)

ax[0,1].scatter(X[y_anomaly==1, 0], X[y_anomaly==1, 1], c='c', edgecolors='k')
ax[0,1].scatter(X[y_anomaly==-1, 0], X[y_anomaly==-1, 1], c='r', marker='x')

# Heatmap of anomaly scores
ax[1,1].scatter(X[:, 0], X[:, 1], c=isoforest.score_samples(X))

XX, YY = np.meshgrid(np.linspace(X[:,0].min(), X[:,0].max(), 100),
                     np.linspace(X[:,1].min(), X[:,1].max(), 100))
Z = isoforest.score_samples(np.c_[XX.ravel(), YY.ravel()]).reshape(XX.shape)
Z = (Z - Z.min()) / (Z.max() - Z.min())
cm = ax[1,1].contourf(XX, YY, Z, levels=50, cmap='RdBu')



---
### Boundary-based methods

> Some approaches (e.g., **One-Class SVMs**) learn a decision boundary that separates normal data points from anomalies. Points that fall outside this boundary are considered anomalies.

For these approaches, it is important to set the expected fraction of anomalies in the dataset $\nu$, as this influences the shape of the decision boundary (as $\nu$ defines how many points can fall outside the learned boundary?)

You can try to apply One-Class SVMs to the previous dataset. Try to change the "kernel" hyperparameter (which defines the shape of the decision boundary). What do the various kernels do?

Next, try to chnge the fraction of anomalies detected. How does it affect the decision boundary? 



---
### Neural Network-based methods

A simple approach is to use Autoencoders, which are neural networks trained to reconstruct their input. A Reconstruction Error (RE) can be computed as the "distance" between the input and the reconstructed output (e.g., Mean Squared Error for continuous variables, Binary Cross Entropy for binary variables, etc.).

#### Autoencoder
It has two parts:  
- Encoder: compresses your input x into a smaller code z
- Decoder: expands z back into a reconstruction \hat{x}

So it learns:  
$x \rightarrow z \rightarrow \hat{x}$  
The bottleneck forces it to learn the main patterns of the data instead of memorizing every detail.  

You train it mostly on normal data (or data that is mostly normal). Then:
- For normal points, the network has seen similar patterns → it reconstructs well → $x \approx \hat{x}$
- For anomalies, the pattern is unfamiliar → it reconstructs poorly → $x and \hat{x} differ a lot$

#### Reconstruction error (RE)
You compute a number that measures “how badly it copied”:
- MSE for continuous features:
$RE = \frac{1}{d}\sum_{j=1}^{d} (x_j - \hat{x}_j)^2$

- Binary cross-entropy for binary features (0/1): measures mismatch in predicted probabilities vs true bits.

> Big RE = suspicious.


#### Thresholding
You still need a rule to decide what RE is “too big”, e.g.:
- top 1% largest RE
- RE > mean + 3·std
- knee on the sorted RE plot

When it fails
- **If you train on data with lots of anomalies, it may learn to reconstruct them too** (then they won’t stand out).
- If you give it too much capacity, it can **memorize everything** → low RE for everything → useless.


In [ ]:
import torch
import torch.nn as nn
from torchvision.datasets import MNIST
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
dataset = MNIST(root='data', train=True, download=True) # only use training set for convenience
X = dataset.data.reshape(-1, 28*28) / 255.0  # normalize to [0, 1] and flatten (==> 60000, 784), i.e., treat each image as a vector of 784 features
print(X.shape)

This is turning MNIST images into a normal “tabular” matrix that models can digest.
- dataset.data is a tensor of shape (60000, 28, 28): 60k grayscale images, each 28×28 pixels. Pixel values are integers 0–255.
- .reshape(-1, 28*28) flattens each 28×28 image into a length-784 vector:
- -1 means “infer this dimension” → it becomes (60000, 784).
- So each row = one image, each column = one pixel position.
- / 255.0 scales pixel intensities from 0–255 to 0–1, which makes training easier (numbers are small and consistent).

In [ ]:
class AutoEncoder(nn.Module):
    def __init__(self):
        # Hardcoding all dimensions for simplicity, we technically could pass them as parameters
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(10, 64),
            nn.ReLU(),
            nn.Linear(64, 256),
            nn.ReLU(),
            nn.Linear(256, 784),
            nn.Sigmoid()  # assuming input is normalized between 0 and 1 (as we previously did)
        )

    def forward(self, x):
        code = self.encoder(x)
        x_rec = self.decoder(code)
        return x_rec

Autoencoder for MNIST where:
- Input x is a 784-dim vector (28×28 flattened image).
- The encoder compresses it down to a 10-dim “code” (bottleneck).
- The decoder expands that 10-dim code back to 784 numbers, aiming to reconstruct the original image.

#### Reminder of NN
The NN is structured as following:
- encoder
- decoder

> The encoder takes an input with dimension 784, compresses it to a dimension of 256, how?
What the model is doing is just this:  
$y = Wx + b$
- $W \in \mathbb{R}^{256 \times 784}$
- $x \in \mathbb{R}^{784}$
- $W \in \mathbb{R}^{256 \times 784},\quad x \in \mathbb{R}^{784 \times 1} \;\;\Rightarrow\;\; y = Wx \in \mathbb{R}^{256 \times 1}$

Since we're multiplying (256 x 784) x (784,1) the output dimension has to be (256, 1) ---> we reduced the output dimension using W

Each component y_j is a dot-product between x and a weight vector w_j (one row of W):
$y_j = w_j^\top x + b_j$

> The model learns a mapping that represents each 784-dim vector using 256 numbers thanks to W ... How is the weight matrix W computed?  
It’s learned by optimization:
1.	Initialize W and b with small random numbers.  
2.	For each training example x:  
- encoder computes $z = f_\theta(x)$ --> the encoded value, which includes W
- decoder produces $\hat{x} = g_\phi(z)$ --> tries to reconstruct the initial input
3.	Compute a loss measuring reconstruction error, e.g.  
$L = \|x - \hat{x}\|^2$  

4.	Compute gradients $\frac{\partial L}{\partial W}$ via backpropagation (chain rule through the whole network).  
5.	Update weights with gradient descent / Adam:  
$W \leftarrow W - \eta \frac{\partial L}{\partial W}$  

6. Repeat over many batches/epochs.  

At the end of training we have W, it's the product of training the model :)

#### What the hell is ReLu?
ReLU is an activation function applied element-by-element:  
$\text{ReLU}(x)=\max(0,x)$  

So:
- if a number is negative → it becomes 0
- if it’s positive → it stays the same

#### Why don't we just directly do nn.Linear(784, 10)?
- We can, but we're forcing the whole image into 10 numbers in one shot.  
- It will usually reconstruct poorly unless the data is extremely simple.  
- Intermediate layers + ReLU let the network learn nonlinear features gradually.

#### Adam optimizer
- Optimizer = the thing that updates the weights W, b to reduce the loss.  
> opt = torch.optim.Adam(model.parameters(), lr=1e-2)  
- Update all model weights using Adam with learning rate 0.01

#### Loss function
- Loss = a single number that says “how bad the model is”.  
- For an autoencoder you want reconstruction \hat{x} close to input x.  
- nn.MSELoss() computes Mean Squared Error:
$\text{MSE}(x,\hat{x}) = \frac{1}{d}\sum_{j=1}^d (x_j - \hat{x}_j)^2$
- So big pixel differences → big loss.

#### TensorDataset
- A Dataset is a PyTorch object that represents your training data and can return items by index.
- It’s just packaging data in the format PyTorch expects.

#### DataLoader
A DataLoader is an iterator that:  
- takes a Dataset
- yields mini-batches
- can shuffle the data

> dl = DataLoader(ds, batch_size=256, shuffle=True)
- each iteration, give me 256 samples; randomize order each epoch

#### Epochs and batches
- Batch (mini-batch): a chunk of data (here 256 samples) used for one weight update.
- Epoch: one full pass over the entire dataset.

If you have 60,000 samples and batch size 256:
- batches per epoch ≈ 60000 / 256 ≈ 235
- so you do ~235 weight updates per epoch

#### sigmoid()
nn.Sigmoid() applies the sigmoid function to each output value:  
$\sigma(x)=\frac{1}{1+e^{-x}}$  
What it does:
- Takes any real number (−∞ to +∞)
- Squashes it into (0, 1)


_--------------_

> Ok, so after this recap we can finally understand what is going on.  
- the encoder:
    - takes a 784 input, compresses it in 256 dimensions
    - performs ReLu
    - compresses 256 dimensions into 64 dimensions
    - performs ReLu
    - compresses 64 dimensions into 10 dimensions
- then the decoder does the opposite
    - from an input of dimension 10 it tries to reconstruct the original output of size 784
- all this stuff is defined in the __init__ of the NN class
- it is performed in the forward method

In [ ]:
model = AutoEncoder()
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.MSELoss()
ds = TensorDataset(X)
dl = DataLoader(ds, batch_size=256, shuffle=True)

n_epochs = 10

for epoch in range(n_epochs):
    epoch_loss = 0.0
    for x_batch, in dl:
        opt.zero_grad()
        batch_rec = model(x_batch)
        loss = loss_fn(batch_rec, x_batch)
        loss.backward()
        opt.step()
        epoch_loss += loss.item()
    epoch_loss /= len(dl)
    print(f"Epoch {epoch+1}/{n_epochs}, Loss: {epoch_loss:.6f}")


1.	opt.zero_grad() clears old gradients (PyTorch accumulates them by default).  
2.	batch_rec = model(x_batch) forward pass (reconstruct).  
3.	loss = loss_fn(batch_rec, x_batch) compute reconstruction error.  
4.	loss.backward() compute gradients of loss w.r.t. all weights (backprop).  
5.	opt.step() update weights using Adam.  
6.	accumulate loss to print an average epoch loss.  

In [ ]:
with torch.no_grad():
    X_rec = model(X).numpy()

X_np = X.numpy()

- with torch.no_grad()
    - Tells PyTorch: “I’m only doing inference, don’t track gradients / don’t build the computation graph.”.
    - Result: less memory, faster, and no accidental backprop.

- X_rec = model(X)
    - Runs the autoencoder forward pass and produces reconstructed outputs (a tensor).

- .numpy()
    - Converts a CPU torch tensor to a NumPy array.

> - reconstruct all inputs, and store the reconstructions as a NumPy array
> - convert the original inputs to NumPy too

In [ ]:
fig, ax = plt.subplots(2, 10, figsize=(15, 3))


for i in range(10):
    ax[0, i].imshow(X_np[i].reshape(28, 28), cmap='gray')
    ax[0, i].set_axis_off()

    ax[1, i].imshow(X_rec[i].reshape(28, 28), cmap='gray')
    ax[1, i].set_axis_off()

The RE (for instance, squared error across all pixels, in this case) can be used to compute how "poorly" each sample is reconstructed by the Autoencoder. High RE values can be used to flag anomalies in the dataset.

Let's look at the worst-reconstructed images according to the MSE RE.

In [ ]:
re = ((X_np - X_rec) ** 2).sum(axis=1)  # Mean Squared Error per sample

fig, ax = plt.subplots(figsize=(6,4))
ax.hist(re, bins=50, color='c', edgecolor='k')
ax.set_ylabel('Frequency')
ax.set_xlabel('Reconstruction Error (Squared Error)')

In [ ]:
fig, ax = plt.subplots(2, 10, figsize=(15, 3))


for i, pos in enumerate(re.argsort()[-10:]):
    ax[0, i].imshow(X_np[pos].reshape(28, 28), cmap='gray')
    ax[0, i].set_axis_off()

    ax[1, i].imshow(X_rec[pos].reshape(28, 28), cmap='gray')
    ax[1, i].set_axis_off()

---

---

# Anomalies detection summary

#### statistical method
- statistical method --> z-score:
    - normalize data: $z = \frac{x - \mu}{\sigma}$ --> distance to the mean expressed in standard deviation
    - fix a threshold, es. 3 standard deviation
    - whatever is further away than the fixed threshold = anomaly  
$|z| > 3 \quad \Longleftrightarrow \quad \frac{|x-\mu|}{\sigma} > 3 \quad \Longleftrightarrow \quad |x-\mu| > 3\sigma$

#### KMeans + DBSCAN
- KMeans + DBSCAN:
    - KMeans:
        - for each point compute distance to closest centroids
        - sort the distances
        - plot the sorted distances --> on the left points closer to the centroids they belong to, on the right points further away from the centroids they belong to = anomalies
        - look for an elbow --> figure out the % of anomalies and compute the number of anomalies N
        - from the sorted distances save the last N distances = anomalies
        - in the plot flag the points having these distances as anomalies
    
    - DBSCAN:
        - already flags outliers / noise / anomalies as lable = -1
        - just plot as usual

#### LOF
- Local Outlier Factor LOF
    - basically identical to DBSCAN, it's density based
    - uses n_neighbours and **contamination**: expected fraction of outliers in the data --> if in doubt go for 0.05
    - returns labels: if labels == 1 the point is fine, if it's ==-1 than it's an outlier
    - **I HAVE NO CLUE HOW TO TUNE THE PARAMETERS**, some ideas might be:
        - Plot the sorted LOF scores **TO CHOOSE CONTAMINATION**:
            - LOF gives each point a “how weird are you” score.
            - Sort those scores
            - plot those scores --> **negative_outlier_factor_**
            - If you see a flat part then a sharp jump (“knee”), that jump is where “normal ends and weird begins”.
            - Set **contamination so you cut around that knee** (or just pick the cutoff score directly instead of using contamination).
        
        - Compare those curves for different n_neighbors **TO CHOOSE N_NEIGHBOURS**:
            - for different n_neighbours like = [10, 20, 35, 50] run the algorithm
            - get the **negative_outlier_factor_**
            - sort them
            - plot them
            - Pick the k where the curve shows the clearest separation (cleanest knee)

#### IsolationForest
- IsolationForest
    - it runs a lot of tree classifier in parallel which randomly select a feature and then randomly slect a split value --> normal data still stays in groups even after random splits, while anomalies get lonly fast = shorter tree length 
    - it's like a normal random forest but with also **contamination**
    - it returns labels, if label == 1 the point is fine, if label == -1 it's an anomaly

#### OneClassSVM
- Boundary based --> OneClassSVM
    - do it yourself

#### NN
- NN
    - it uses autoencoders
    - you pass in input X with dimension N to an encoder, which reduces the dimension on the input compressing it to K << N
    - then a decoder tries to reconstruct the original input $\hat{X}$
    - build the model, the optimizer, the loss function, the tensor dataset, the dataloader and fix the number of epochs
    - train the model:
        - iterate on epochs
        - iterate on batches
        - zero the optimizer gradients
        - do the forward pass of the NN model passing the input X to compute the reconstructed input $\hat{X}$
        - compute loss function
        - take a step with otpimizer
    - then we compute reconstruction error RE for each feature:
        - if the input is a continuous number we use MSE
        - if the input is binary we use binary cross entropy
    - higher RE == anomalies